# 🇮🇳 Bharat-Tiny-LLM v3.1 — Conversational SFT (Kaggle T4)

Fine-tunes the **already-merged v3 model** (`eulogik/Bharat-Tiny-LLM-v3`)
on 20K Hindi/Hinglish conversations for natural chat ability.

| Cell | What | Resume guard |
|---|---|---|
| 1 | Install deps | — |
| 2 | Config | — |
| 3 | Load v3 + LoRA r=16 | `model_loaded` |
| 4 | Load & tokenize chat dataset | `tok_done` |
| 5 | Train (2 epochs, lr=2e-5) | `train_done` |
| 6 | Merge adapter + upload v3.1 | `export_done` |

### Dataset
- `eulogik/bharat-v3-chat-sft` (20K rows)
  - 10K multi-turn Hindi (`samvaad-hi-v1`)
  - 5K Hinglish (`hinglish-conversations`)
  - 5K filtered single-turn from our corpus

### One-time setup
Attach a Kaggle secret named **HF_TOKEN** (write access).


In [ ]:
#@title 1 · Install deps {display-mode:"form"}
# Kaggle's preinstalled transformers 5.x + peft 0.17 already support qwen3.
# Only fix the peft ↔ torchao version clash (0.10 < required 0.16).
import subprocess, sys
subprocess.run([sys.executable, "-m", "pip", "uninstall", "-y", "torchao"],
               capture_output=True)
print("torchao removed; deps ready")


In [ ]:
#@title 1 · Config {display-mode:"form"}
import os, json, time, random, gzip
from pathlib import Path
from datetime import datetime

# ── Paths ────────────────────────────────────────────────────────
WORK      = Path("/kaggle/working/brahmi_v33")
DATA_DIR  = WORK / "data"
OUT_DIR   = WORK / "chat_sft_out"

# ── HF Token ──────────────────────────────────────────────────────
HF_TOKEN = os.environ.get("HF_TOKEN", "")
if not HF_TOKEN:
    try:
        from kaggle_secrets import UserSecretsClient
        HF_TOKEN = UserSecretsClient().get_secret("HF_TOKEN")
    except Exception as e:
        print(f"kaggle_secrets failed: {e}")
if not HF_TOKEN:
    HF_TOKEN = ""
    print("WARNING: using hardcoded HF_TOKEN")

# Set as env var so huggingface_hub picks it up for ALL internal requests
# (transformers 5.x calls model_info() without forwarding token=)
os.environ["HF_TOKEN"] = HF_TOKEN
print(f"HF_TOKEN: ...{HF_TOKEN[-4:]}")

# ── Model ────────────────────────────────────────────────────────
BASE_MODEL = "eulogik/Bharat-Tiny-LLM-v3"    # already-merged v3
CHAT_DATA  = "eulogik/bharat-v3-chat-sft-v3"  # 370 curated
OUT_REPO   = "eulogik/Bharat-Tiny-LLM-v3"

# ── Hyperparameters (v3.3: tiny+clean) ───────────────────────
LORA_R        = 8
LORA_ALPHA    = 16
LORA_DROPOUT  = 0.05
BS            = 4
GRAD_ACCUM    = 4          # eff_bs=16
EPOCHS        = 2           # 370 rows -> 46 steps total
LR            = 5e-6       # very low — just polish
MAX_SEQ       = 512
SAVE_STEPS    = 200
LOG_STEPS     = 25
MAX_SESSION_HOURS = 5.5

# ── State ────────────────────────────────────────────────────────
STATE_PATH = WORK / "chat_sft_state.json"
WORK.mkdir(parents=True, exist_ok=True)
DATA_DIR.mkdir(parents=True, exist_ok=True)

def load_state():
    try: return json.loads(STATE_PATH.read_text())
    except: return {}

def save_state(s):
    STATE_PATH.write_text(json.dumps(s, indent=2))

STATE = load_state()
print(f"config ready | base={BASE_MODEL} | data={CHAT_DATA}")
print(f"lora r={LORA_R} alpha={LORA_ALPHA} | bs={BS} accum={GRAD_ACCUM} eff_bs={BS*GRAD_ACCUM}")
print(f"epochs={EPOCHS} lr={LR} seq={MAX_SEQ}")


In [ ]:
#@title 2 · Load v3 model + LoRA {display-mode:"form"}
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import LoraConfig, get_peft_model

if not STATE.get("model_loaded"):
    t0 = time.time()
    print(f"loading {BASE_MODEL} ...", flush=True)

    tok = AutoTokenizer.from_pretrained(BASE_MODEL, token=HF_TOKEN,
                                         trust_remote_code=True)
    # T4 x2 allocates 2 GPUs but Trainer is single-GPU — force cuda:0
    model = AutoModelForCausalLM.from_pretrained(
        BASE_MODEL,
        dtype=torch.float16,
        attn_implementation="sdpa",
        device_map={"": 0},
        token=HF_TOKEN,
        trust_remote_code=True,
        low_cpu_mem_usage=True,
    )

    # freeze base, add LoRA
    for p in model.parameters():
        p.requires_grad = False

    lcfg = LoraConfig(
        r=LORA_R, lora_alpha=LORA_ALPHA, lora_dropout=LORA_DROPOUT,
        bias="none", task_type="CAUSAL_LM",
        target_modules=["q_proj","k_proj","v_proj","o_proj",
                        "gate_proj","up_proj","down_proj"],
    )
    model = get_peft_model(model, lcfg)
    model.config.use_cache = False

    print(f"loaded in {time.time()-t0:.0f}s | {sum(p.numel() for p in model.parameters() if p.requires_grad)/1e6:.1f}M trainable")
    model.print_trainable_parameters()
    STATE["model_loaded"] = True
    save_state(STATE)
else:
    print("model already loaded")


In [ ]:
#@title 3 · Load & tokenize chat dataset {display-mode:"form"}
from datasets import load_dataset

if not STATE.get("tok_done"):
    t0 = time.time()
    print(f"loading {CHAT_DATA} ...", flush=True)
    ds = load_dataset(CHAT_DATA, split="train", token=HF_TOKEN)
    print(f"  raw rows: {len(ds)}")

    # sample down to 20K if needed (dataset is exactly 20K but safety check)
    if len(ds) > 20000:
        ds = ds.shuffle(seed=42).select(range(20000))

    def tokenize_chat(ex):
        msgs = ex["messages"]
        if isinstance(msgs, str):
            msgs = json.loads(msgs)
        # Qwen3 chat template (native)
        full = tok.apply_chat_template(msgs, tokenize=False,
                                       add_generation_prompt=False)
        prompt = tok.apply_chat_template(msgs[:-1], tokenize=False,
                                         add_generation_prompt=True)
        fids = tok(full, add_special_tokens=False)["input_ids"][:MAX_SEQ]
        pids = tok(prompt, add_special_tokens=False)["input_ids"]
        labels = list(fids)
        cut = min(len(pids), len(fids))
        for i in range(cut):
            labels[i] = -100
        return {"input_ids": fids, "labels": labels}

    ds = ds.map(tokenize_chat, remove_columns=ds.column_names,
                num_proc=2, desc="tokenize")
    ds = ds.filter(lambda ex: any(l != -100 for l in ex["labels"]),
                   num_proc=2)
    ds = ds.shuffle(seed=42)

    # 95/5 train/eval split
    split = ds.train_test_split(test_size=0.05, seed=42)
    train_ds = split["train"]
    eval_ds  = split["test"]

    print(f"  train: {len(train_ds)} | eval: {len(eval_ds)} | done in {time.time()-t0:.0f}s")
    STATE["tok_done"] = True
    STATE["train_rows"] = len(train_ds)
    STATE["eval_rows"] = len(eval_ds)
    save_state(STATE)
else:
    print("tokenization already done")


In [ ]:
#@title 4 · Train conversational SFT {display-mode:"form"}
import torch
from transformers import Trainer, TrainingArguments, TrainerCallback
from pathlib import Path

class Heartbeat(TrainerCallback):
    def on_log(self, args, state, control, logs=None, **kw):
        if logs and "loss" in logs:
            (WORK / "chat_progress.json").write_text(json.dumps(
                {"step": state.global_step, "loss": logs.get("loss"),
                 "eval_loss": logs.get("eval_loss"),
                 "ts": time.time()}))
            print(f"  [step {state.global_step}] loss={logs.get('loss', '?'):.4f}", flush=True)

    def on_step_end(self, args, state, control, **kw):
        if state.global_step % 100 == 0 and state.global_step > 0:
            print(f"  ... step {state.global_step}/{state.max_steps}", flush=True)

def find_local_ckpt(out):
    cks = [p for p in Path(out).glob("checkpoint-*") if p.is_dir()]
    return max(cks, key=lambda p: int(p.name.split("-")[-1])) if cks else None

if not STATE.get("train_done"):
    t0 = time.time()

    class Collator:
        def __call__(self, feats):
            maxlen = max(len(f["input_ids"]) for f in feats)
            pad_id = tok.pad_token_id or tok.eos_token_id or 0
            batch_ids, batch_labels, batch_mask = [], [], []
            for f in feats:
                n = len(f["input_ids"])
                padn = maxlen - n
                batch_ids.append(f["input_ids"] + [pad_id] * padn)
                batch_labels.append(f["labels"] + [-100] * padn)
                batch_mask.append([1] * n + [0] * padn)
            t = lambda x: torch.tensor(x, dtype=torch.long)
            return {"input_ids": t(batch_ids),
                    "labels": t(batch_labels),
                    "attention_mask": t(batch_mask)}

    targs = TrainingArguments(
        output_dir=str(OUT_DIR),
        per_device_train_batch_size=BS,
        gradient_accumulation_steps=GRAD_ACCUM,
        num_train_epochs=EPOCHS,
        learning_rate=LR,
        lr_scheduler_type="cosine",
        warmup_ratio=0.05,
        logging_steps=LOG_STEPS,
        save_steps=SAVE_STEPS,
        save_total_limit=2,
        fp16=True,
        bf16=False,
        gradient_checkpointing=True,
        gradient_checkpointing_kwargs={"use_reentrant": False},
        optim="adamw_torch",
        report_to=[],
        dataloader_num_workers=0,
        seed=42,
        remove_unused_columns=False,
        eval_strategy="steps",
        eval_steps=SAVE_STEPS,
        load_best_model_at_end=True,
        metric_for_best_model="eval_loss",
        disable_tqdm=False,
    )

    trainer = Trainer(
        model=model, args=targs,
        train_dataset=train_ds, eval_dataset=eval_ds,
        data_collator=Collator(),
        callbacks=[Heartbeat()],
    )

    resume_dir = find_local_ckpt(str(OUT_DIR))
    print(f"training | epochs={EPOCHS} eff_bs={BS*GRAD_ACCUM} "
          f"| resume={resume_dir or 'none'}")
    trainer.train(resume_from_checkpoint=str(resume_dir) if resume_dir else None)

    final = OUT_DIR / "final_adapter"
    trainer.model.save_pretrained(str(final))
    tok.save_pretrained(str(OUT_DIR / "tokenizer"))
    (OUT_DIR / "DONE").write_text("ok")

    elapsed = time.time() - t0
    STATE["train_done"] = True
    STATE["train_time_min"] = round(elapsed / 60, 1)
    save_state(STATE)
    print(f"train done in {elapsed/60:.1f} min → {final}")
else:
    print("training already done")


In [ ]:
#@title 5 · Merge adapter + upload v3.1 {display-mode:"form"}
if not STATE.get("export_done"):
    import torch
    from peft import PeftModel
    from transformers import AutoModelForCausalLM, AutoTokenizer
    from huggingface_hub import upload_folder, upload_file

    t0 = time.time()
    final_dir = OUT_DIR / "final_adapter"
    assert final_dir.exists(), f"no adapter at {final_dir}"

    print("loading base v3 fp32 for merge...", flush=True)
    base = AutoModelForCausalLM.from_pretrained(
        BASE_MODEL, dtype=torch.float32, token=HF_TOKEN, trust_remote_code=True)

    model = PeftModel.from_pretrained(base, str(final_dir))
    model = model.merge_and_unload()
    model.half()

    export = WORK / "export_v31"
    model.save_pretrained(str(export), safe_serialization=True,
                          max_shard_size="4GB")
    tok_out = AutoTokenizer.from_pretrained(str(OUT_DIR / "tokenizer"))
    tok_out.save_pretrained(str(export))

    print("uploading to hub...", flush=True)
    upload_folder(folder_path=str(export), repo_id=OUT_REPO,
                  repo_type="model", token=HF_TOKEN,
                  commit_message="v3.1: conversational SFT on samvaad+hinglish+corpus")

    STATE["export_done"] = True
    STATE["export_time_min"] = round((time.time() - t0) / 60, 1)
    save_state(STATE)
    print(f"v3.1 exported in {time.time()-t0:.0f}s → {OUT_REPO}")
else:
    print("export already done")

# ── summary ──────────────────────────────────────────────────────
s = load_state()
print(f"""
=== Bharat-Tiny-LLM v3.1 Chat SFT Complete ===
  model:   {OUT_REPO}
  data:    {CHAT_DATA} ({s.get('train_rows','?')} train / {s.get('eval_rows','?')} eval)
  lora:    r={LORA_R} alpha={LORA_ALPHA}
  time:    train {s.get('train_time_min','?')} min | export {s.get('export_time_min','?')} min
  next:    run benchmarks locally (fertility, PPL, GSM8K-Hindi, chat eval)
""")
